**Group 5:**
1. Lucia Francescoli
2. Frank Novoa
3. Maxime Engels
4. Tiziano Gallo




**Problem Description:**

The Japanese higher education system is highly competitive and internationally recognized. Students who wish to study in Japan face complex decisions, considering factors such as internationalization of the university, research impact, and graduate employability.


**Research Question:**

which universities and regions in Japan offer the best opportunities for international students, considering internationalization, research impact, and employability, and to detect geographic gaps or inequalities that could hinder the attraction of global talent.

**Stakeholders:**

International students

**About Datasets**

1. **Japan Top 50 Universities Dataset:** This dataset tracks the academic and professional performance of the top 52 universities in Japan.

      **Variables:**
      * *`University_Name:`* The recognized international English name.
      * *`National_Rank:`* Comprehensive 2026 rank based on academic reputation, employer reputation, and faculty/student ratio.
      * *`Founded_Year:`* The year of original founding (or the effective date of a major structural merger).
      * *`Institution_Type`*: National (Government-funded), Public (Prefectural/Municipal), or Private.
      * *`Region_code:`* The geographic prefecture code
      * *`Research_Impact_Score:`* A weighted metric (0-100) analyzing citations per faculty and research grants.
      * *`Intl_Student_Ratio:`* Percentage of non-Japanese degree-seeking students.
      * *`Employment_Rate:`* The percentage of graduates entering full-time employment or higher-level graduate studies.

      **Link:** https://www.kaggle.com/datasets/nudratabbas/japan-top-50-universities-dataset

2. **Japan Region Dataset:** This dataset provides basic demographic and geographic information for all 47 prefectures of Japan in 2024.

      **Variables:**
      * *`Region_code:`* The geographic prefecture code
      * *`Region:`* The name of the prefecture/region in Japan
      * *`Total_Population:`* Total population (Both sexes) in 2024

2. **Japan Expenditure Dataset:** This dataset provides basic expenditures information for all 47 prefectures of Japan in 2007.

      **Variables:**
      * *`Region_code:`* The geographic prefecture code
      * *`Region:`* The name of the prefecture/region in Japan
      * *`Living_Expenditure:`* Average living expenditures in Yen in 2007
      * *`Housing_Expenditure:`* Average housing expenditure in Yen in 2007
      * *`Education_Expenditure:`* Average education expenditure in 2007

      **Link:** https://www.e-stat.go.jp/en/regional-statistics/ssdsview/prefectures

In [ ]:
!pip install pyspark
!pip install plotly

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("Japan_Unis") \
    .getOrCreate()

In [ ]:
import plotly.express as px # We chose this library to create dynamic visualizations with hover information
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# **Importing CSV files**

**Option 1: Running the code in Google Colab**

Select the following datasets:

1. japan_universities_2026.csv
2. region_japan_2024.csv
3. expenditure_region.csv



In [ ]:
from google.colab import files
uploaded = files.upload()

df_ranking = spark.read.csv("/content/japan_universities_2026.csv", header=True, inferSchema=True)

df_region = spark.read.csv("/content/region_japan_2024.csv", header=True, inferSchema=True)

df_expenditure = spark.read.csv("/content/expenditure_region.csv", header=True, inferSchema=True)

df_region_analysis = df_ranking.join(df_region, "Region_Code")\
    .groupBy("Region", "Total_Population")\
    .agg(
        count("*").alias("Num_Universities"),
        avg("Intl_Student_Ratio").alias("Avg_Intl_Ratio")
    )\
    .withColumn("Universities_per_Million",
                round(col("Num_Universities") / (col("Total_Population")/1000000), 2))


**Option 2: Running the code locally**

In [ ]:

df_ranking = spark.read.csv("japan_universities_2026.csv", header=True, inferSchema=True)

df_region = spark.read.csv("region_japan_2024.csv", header=True, inferSchema=True)

df_expenditure = spark.read.csv("expenditure_region.csv", header=True, inferSchema=True)

df_region_analysis = df_ranking.join(df_region, "Region_Code")\
    .groupBy("Region", "Total_Population")\
    .agg(
        count("*").alias("Num_Universities"),
        avg("Intl_Student_Ratio").alias("Avg_Intl_Ratio")
    )\
    .withColumn("Universities_per_Million",
                round(col("Num_Universities") / (col("Total_Population")/1000000), 2))


In [ ]:
df_region_analysis.show()

In [ ]:
df_ranking.printSchema()

In [ ]:
df_ranking.show()

In [ ]:
df_region.printSchema()

In [ ]:
df_region.show()

In [ ]:
df_region_stats = df_ranking.join(
    df_region,
    on="Region_Code"
).groupBy("Region").agg(
    avg("Intl_Student_Ratio").alias("Avg_Intl"),
    avg("Employment_Rate").alias("Avg_Employment"),
    avg("Research_Impact_Score").alias("Avg_Research")
)

In [ ]:
df_region_stats.show()

**1. Find the top 3 universities with the highest proportion of international students.**

In [ ]:
df_ranking_top3 = df_ranking.orderBy(df_ranking.Intl_Student_Ratio.desc()).limit(3)

ranking_top3 = df_ranking_top3.toPandas()

fig = px.bar(
    ranking_top3,
    x='University_Name',
    y='Intl_Student_Ratio',
    color='Intl_Student_Ratio',
    text_auto='.1f',
    title='Top 3 Universities by International Student Ratio in Japan',
    color_continuous_scale=['#264653', '#2a9d8f']
)

text_color = "#334155"

fig.update_layout(
    plot_bgcolor='white',
    font=dict(family="Verdana", size=12, color=text_color),
    xaxis_title="",
    yaxis_title="International Student Ratio (%)",
    coloraxis_showscale=False,
    margin=dict(l=60, r=50, t=80, b=50),
    height=500,

    xaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=False
    ),
    yaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=True,
        gridcolor='#f0f0f0'
    )
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=1,
    textposition='outside',
    textfont=dict(family="Verdana Bold", color=text_color),
    cliponaxis=False
)

fig.show()

**Conclusion:** Ritsumeikan Asia Pacific University has the highest proportion of international students. Followed by Akita International University and International Chistrian University.

**2. Find the region with the highest number of universities.**

In [ ]:
df_joined = df_ranking.join(
    df_region,
    df_ranking.Region_Code == df_region.Region_Code
).select(
    "University_Name",
    df_region.Region,
)

region_counts = df_joined.groupBy("Region").agg(count("*")).withColumnRenamed("count(1)", "Num_Universities")
region_counts.show()

In [ ]:
max_num_univ_reg = region_counts.groupBy().max("Num_Universities").take(1)[0][0]

print(max_num_univ_reg)

df_max_num_univ_reg=region_counts.where(
    region_counts.Num_Universities == max_num_univ_reg
)
df_max_num_univ_reg.show()

In [ ]:
region_count = region_counts.toPandas().sort_values('Num_Universities', ascending=False)

fig = px.bar(
    region_count,
    x='Region',
    y='Num_Universities',
    color='Num_Universities',
    text_auto=True,
    title='Universities by Region',
    color_continuous_scale=[
        (0, '#264653'),
        (0.1, '#2a9d8f'),
        (1, '#e9c46a')
    ]
)

text_color = "#334155"

fig.update_layout(
    plot_bgcolor='white',
    font=dict(family="Verdana", size=12, color=text_color),
    xaxis_title="",
    yaxis_title="Total Universities",
    coloraxis_showscale=False,
    margin=dict(l=50, r=50, t=80, b=100),
    height=600,

    xaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=False
    ),
    yaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=True,
        gridcolor='#f0f0f0'
    )
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.5,
    textposition='outside',
    textfont=dict(family="Verdana Bold", color=text_color),
    cliponaxis=False
)

fig.update_xaxes(tickangle=45)

fig.show()

**Conclusion:** A high number of universities means more program variety, courses, and study environments, making Tokyo attractive for students from abroad.

**3. For each type of institution, calculate the average proportion of international students and count the number of universities**

In [ ]:
df_type = df_ranking.groupBy("Institution_Type").agg(
    count("*"),
    avg("Intl_Student_Ratio").alias("Avg_Intl_Student_Ratio"),
).orderBy("Avg_Intl_Student_Ratio", ascending=False)\
.withColumn("Avg_Intl_Student_Ratio", round("Avg_Intl_Student_Ratio", 2))\
.withColumnRenamed("count(1)", "Num_Universities")

df_type.show()

In [ ]:
type_pd = df_type.toPandas().sort_values('Avg_Intl_Student_Ratio', ascending=False)

fig = px.bar(
    type_pd,
    x='Institution_Type',
    y='Avg_Intl_Student_Ratio',
    color='Avg_Intl_Student_Ratio',
    text_auto='.1f',
    title='Average International Student Ratio by Institution Type',
    color_continuous_scale=['#264653', '#2a9d8f']
)

text_color = "#334155"

fig.update_layout(
    plot_bgcolor='white',
    font=dict(family="Verdana", size=12, color=text_color),
    xaxis_title="",
    yaxis_title="Average International Student Ratio (%)",
    coloraxis_showscale=False,
    margin=dict(l=60, r=50, t=80, b=50),
    height=500,

    xaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=False
    ),
    yaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=True,
        gridcolor='#f0f0f0'
    )
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=1,
    textposition='outside',
    textfont=dict(family="Verdana Bold", color=text_color),
    cliponaxis=False
)


fig.show()

**Conclusion:** Public universities, although few in number, have a higher average proportion of international students compared to private and national universities. This suggests that, relatively, public universities are more open or attractive to international students.

**4. Find the top 10 universities in Japan that provide the best graduate employability for students.**

In [ ]:
top10_employment = df_ranking.select(
    'University_Name',
    'Institution_Type',
    'Employment_Rate'
).orderBy(
    col("Employment_Rate").desc()
).limit(10)
top10_employment.show()

In [ ]:
top_employ = top10_employment.toPandas().sort_values(
    by=["Employment_Rate", "Institution_Type"],
    ascending=[False, True]
)

fig = px.bar(
    top_employ,
    x="Employment_Rate",
    y="University_Name",
    color="Institution_Type",
    orientation='h',
    text_auto='.1f',
    title='Top 10 Universities by Graduate Employability in Japan',
    color_discrete_map={'National': '#2a9d8f', 'Private': '#264653'}
)

fig.update_layout(
    plot_bgcolor='white',
    xaxis_title="Employment Rate (%)",
    yaxis_title="",
    yaxis={'categoryorder':'array', 'categoryarray': top_employ['University_Name'][::-1]},
    xaxis=dict(
        range=[95, 99],
        showticklabels=False,
        showgrid=False,
        zeroline=False
    ),
    legend_title="Institution Type",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=200, r=50, t=100, b=50),
    height=600
)

fig.update_traces(
    marker_line_color='black',
    marker_line_width=1,
    textposition='outside',
    cliponaxis=False
)

fig.show()

**Conclusion:** Private and national universities in Japan tend to have strong industry connections and higher graduate employability rates. In contrast, public universities do not appear among the top-ranked institutions for employability.

**5. Find the top 5 universities in Japan that offer the highest overall opportunity for international students, considering research impact, employability, and internationalization.**

In [ ]:
def opportunity_score(research, employment, intl_ratio):
    """
    Calculates a simple opportunity score for international students
    by combining research impact, employability, and internationalization.
    For this we considered research, employment and international ratio as equaly important for students.
    """
    return (research + employment + intl_ratio) / 3

In [ ]:
opportunity_udf = udf(opportunity_score, DoubleType())

df_scores = df_ranking.withColumn(
    "Opportunity_Score",
    opportunity_udf(
        df_ranking.Research_Impact_Score,
        df_ranking.Employment_Rate,
        df_ranking.Intl_Student_Ratio
    )
)

df_scores_university = df_scores.orderBy(col("Opportunity_Score").desc()).select(
    "University_Name", "Opportunity_Score"
).limit(5)

df_scores_university.show()

In [ ]:
top_score_pd = df_scores_university.select("University_Name", "Opportunity_Score").toPandas()

fig = px.bar(
    top_score_pd,
    x='University_Name',
    y='Opportunity_Score',
    text_auto='.1f',
    title='Top 5 Universities in Japan by Opportunity Score',
    color='Opportunity_Score',
    color_continuous_scale='sunset'
)

text_color = "#334155"

fig.update_layout(
    xaxis_title="",
    yaxis_title="Opportunity Score",
    yaxis_range=[60, 71],
    plot_bgcolor='white',
    font=dict(family="Verdana", size=12, color=text_color),
    title_font=dict(size=18),
    coloraxis_showscale=False,

    xaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=False
    ),
    yaxis=dict(
        showline=True,
        linecolor=text_color,
        linewidth=1,
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    margin=dict(l=60, r=50, t=80, b=50)
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=1,
    textposition='outside',
    textfont=dict(family="Verdana Bold", color=text_color),
    cliponaxis=False
)


fig.show()

**Conclusion:** The University of Tokyo has the highest Opportunity Score among Japanese universities. This indicates that it offers the best overall opportunities for international students seeking quality education, career prospects, and an international environment.

**6. Find the regions with highest inequality.**



In [ ]:
gap_pd = df_region_analysis.orderBy(col("Universities_per_Million").desc()).toPandas()

fig = px.bar(
    gap_pd,
    x='Universities_per_Million',
    y='Region',
    orientation='h',
    text_auto='.2f',
    title='Geographic Inequality: Regions with Lowest University Density',
    color='Universities_per_Million',
    color_continuous_scale='viridis'
)

fig.update_layout(
    height=1000,
    plot_bgcolor='white',
    xaxis_title="Number of Top Universities per Million People",
    yaxis_title="",
    title_font=dict(size=20, family="Arial", color="black"),
    coloraxis_showscale=False,
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    margin=dict(l=150, r=80, t=80, b=50)
)

fig.update_traces(
    textposition='outside',
    marker_line_color='black',
    marker_line_width=1,
    textfont=dict(size=11, color="black", family="Arial"),
    cliponaxis=False
)

fig.show()

**Conclusion:** Saitama is the region with the most significant geographic inequality, having the lowest university density relative to its population.

**7. Find if there is a relation between a high international student body and employability in Japanese universities**



In [ ]:
df_all_scores = df_ranking.withColumn(
    "Opportunity_Score",
    opportunity_udf(
        col("Research_Impact_Score"),
        col("Employment_Rate"),
        col("Intl_Student_Ratio")
    )
)

df_final = df_all_scores.select(
    "University_Name",
    "Employment_Rate",
    "Intl_Student_Ratio",
    "Opportunity_Score",
    "Institution_Type"
)

df_plot = df_final.toPandas()

# Scatter plot to see if higher number of international students correlates to job prospects across Japan.
fig = px.scatter(
    df_plot,
    x='Employment_Rate',
    y='Intl_Student_Ratio',
    size='Opportunity_Score',
    color='Institution_Type',
    hover_name='University_Name',
    title='Graduate Employability vs. International Student Body',
    labels={
        'Employment_Rate': 'Employability Rate (%)',
        'Intl_Student_Ratio': 'International Student Ratio (%)'
    },
    color_discrete_map={'National': '#2a9d8f', 'Private': '#264653', 'Public': '#72efdd'},
    opacity=0.7
)

text_color = "#334155"
fig.update_layout(
    plot_bgcolor='white',
    font=dict(family="Verdana", size=12, color=text_color),
    margin=dict(l=60, r=50, t=80, b=50),
    height=600,
    xaxis=dict(showline=True, linecolor=text_color, linewidth=1, showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(showline=True, linecolor=text_color, linewidth=1, showgrid=True, gridcolor='#f0f0f0'),
    legend_title="Institution Type"
)

avg_emp = df_plot['Employment_Rate'].mean()
avg_intl = df_plot['Intl_Student_Ratio'].mean()

fig.add_vline(x=avg_emp, line_dash="dash", line_color="#94a3b8",
              annotation_text="Avg Employability", annotation_position="top left")
fig.add_hline(y=avg_intl, line_dash="dash", line_color="#94a3b8",
              annotation_text="Avg Intl Ratio", annotation_position="bottom right")

fig.show()

**Conclusion:** Universities with fewer international students usually have lower employment rates, and national universities seem to be a top choice because they have both many international students and great job prospects.

**8. Find the top 10 Universities sorted by lowest living expenditures**


In [ ]:
df_ranking.printSchema()

In [ ]:
df_expenditure.printSchema()

In [ ]:
#Join the two datasets
df_joined = df_ranking.join(df_expenditure, on='Region_Code', how='inner')

df_joined.printSchema()
df_joined.show(5)

In [ ]:
df_cleaned = df_joined.withColumn("Living_Expenditure_Yen_Cleaned",
                                    regexp_replace(col("Living_Expenditure_Yen"), ",", "").cast(IntegerType()))

In [ ]:
top_universities_by_expenditure = (df_cleaned.orderBy(col("National_Rank")).limit(10)
                                   .orderBy(col("Living_Expenditure_Yen_Cleaned"))
                                   .select("Region", "University_Name", "National_Rank", "Living_Expenditure_Yen_Cleaned"))

top_universities_by_expenditure.show(truncate=False)

In [ ]:
df_plot = top_universities_by_expenditure.toPandas()
df_plot['University_Name_Rank'] = (
    df_plot['National_Rank'].astype(str) + '. ' + df_plot['University_Name']
)
df_plot = df_plot.sort_values('Living_Expenditure_Yen_Cleaned')

y_min = df_plot['Living_Expenditure_Yen_Cleaned'].min() * 0.95
y_max = df_plot['Living_Expenditure_Yen_Cleaned'].max() * 1.05

fig = px.bar(
    df_plot,
    x='University_Name_Rank',
    y='Living_Expenditure_Yen_Cleaned',
    text='Living_Expenditure_Yen_Cleaned',
    color='Living_Expenditure_Yen_Cleaned',
    color_continuous_scale='viridis'
)

fig.update_layout(
    height=800,
    plot_bgcolor='white',
    xaxis_title="University Name (Rank)",
    yaxis_title="Living Expenditure (Yen)",
    title='Top Universities by National Rank, Sorted by Lowest Living Expenditure',
    title_font=dict(size=20, family="Arial", color="black"),
    margin=dict(l=100, r=50, t=80, b=200),
    coloraxis_showscale=False,
    xaxis=dict(
        tickangle=60,
        showgrid=False,
        showline=True,
        linecolor='black',
        linewidth=2
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        zeroline=False,
        range=[y_min, y_max],
        showline=True,
        linecolor='black',
        linewidth=2
    )
)

fig.update_traces(
    texttemplate='%{y:,.0f}',
    textposition='outside',
    marker_line_color='black',
    marker_line_width=1,
    textfont=dict(size=11, color="black", family="Arial"),
    cliponaxis=False
)

fig.show()

**Conclusion:** Top-ranked universities outside Tokyo are cheaper to live in than Tokyo-based ones. High academic quality does not require high living expenditure.


**9. Compare the top 10 universities with the highest employment rate and their living expenditure**




In [ ]:
top_universities_employment_by_expenditure = (df_cleaned.orderBy(col("Employment_Rate").desc()).limit(10)
                                                    .select("Region", "University_Name", "National_Rank", "Employment_Rate", "Living_Expenditure_Yen_Cleaned"))

top_universities_employment_by_expenditure.show(truncate=False)

In [ ]:
df_plot = top_universities_employment_by_expenditure.toPandas()

fig = px.scatter(
    df_plot,
    x='Living_Expenditure_Yen_Cleaned',
    y='Employment_Rate',
    color='University_Name',
    size='National_Rank',
    size_max=50,
    color_continuous_scale='viridis',
    hover_name='University_Name',
)

fig.update_traces(
    marker=dict(
        line=dict(width=1, color='black'),
        opacity=0.8
    )
)

fig.update_layout(
    plot_bgcolor='white',
    title='Employment Rate vs. Living Expenditure for Top Universities',
    title_font=dict(size=20, family="Arial", color="black"),
    xaxis_title='Living Expenditure (Yen)',
    yaxis_title='Employment Rate (%)',
    xaxis=dict(showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey'),
    legend=dict(
        title='University Name',
        x=1.05,
        y=1,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(0,0,0,0)'
    ),
    height=700,
    width=1000
)

fig.show()

**Conclusion:** Employment rates remain high across universities, regardless of living expenditure, and higher living expenditure does not guarantee better employment outcomes.